# Trabajo Práctico: Seguimiento y caracterización de satélites
**Integrantes:** Franco Diz y Lautaro Mesa  
**Descripción:** Informe de la Etapa 1 enfocado en el análisis estático de la población satelital actual a partir del catálogo CelesTrak.

## 1. Carga de datos y estado inicial

En esta sección se cargan los registros astronómicos desde el archivo JSON, se calcula la fecha media del catálogo (`EPOCH`) para estimar la antigüedad de los datos, y se filtran aquellos objetos cuya altitud es menor a 100 km (considerados decaídos o fuera de órbita operativa).

In [1]:
from datetime import datetime, timezone

from catalogo import leer_satelites
from satelite import crear_satelites
from funciones_base import epoch_medio

# 1. Leer los registros y guardar el momento actual (en UTC)
ruta_json = "datos/satelites.json"
registros = leer_satelites(ruta_json)
ahora = datetime.now(timezone.utc)

# 2. Crear los objetos Satelite descartando los de altitud < 100 km
satelites = crear_satelites(registros, ahora)

# 3. Calcular métricas solicitadas
epoch_dt = epoch_medio(registros)
dias_transcurridos = (ahora - epoch_dt).days
descartados = len(registros) - len(satelites)

# Imprimir resultados
print(f"Registros totales leídos: {len(registros)}")
print(f"Fecha media del EPOCH (datos): {epoch_dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Días transcurridos desde el EPOCH medio: {dias_transcurridos} días")
print(f"Satélites descartados (altitud < 100 km): {descartados}")
print(f"Satélites válidos procesados: {len(satelites)}")

Registros totales leídos: 16710
Fecha media del EPOCH (datos): 2026-10-14 06:30:18 UTC
Días transcurridos desde el EPOCH medio: 12 días
Satélites descartados (altitud < 100 km): 16
Satélites válidos procesados: 16694


**Lectura:** el archivo tiene 16.710 satélites y sus órbitas se midieron, en promedio, hace unos 12 días. Esa antigüedad es poca, así que las posiciones calculadas para hoy son confiables (el error crece cuanto más lejos de esa fecha se calcula). Se descartaron unos pocos satélites cuya altitud daba menos de 100 km: ya cayeron a la atmósfera y el cálculo no tiene sentido para ellos.

## 2. Grupos

Se cuenta cuántos satélites pertenecen a cada constelación según el prefijo de su nombre, y se calcula qué porcentaje del total es Starlink.

In [2]:
from catalogo import contar_por_grupo

conteo_grupos = contar_por_grupo(registros)
for nombre, cantidad in conteo_grupos.items():
    print(f"{nombre}: {cantidad}")

porcentaje_starlink = conteo_grupos["Starlink"] / len(registros) * 100
print(f"Starlink representa el {porcentaje_starlink:.1f}% del total")

Otros: 4075
GPS: 40
Globalstar: 36
BeiDou: 55
Iridium: 80
OneWeb: 651
Starlink: 11125
Qianfan: 257
Kuiper: 391
Starlink representa el 66.6% del total


**Lectura:** Starlink domina el catálogo: tiene 11.125 satélites, alrededor de dos tercios del total (66,6 %). Después vienen OneWeb (651), Kuiper (391) y Qianfan (257), y el resto de las constelaciones tiene menos de 100 satélites cada una. Hay además 4.075 satélites que no pertenecen a ninguno de estos grupos ("Otros").

## 3. Lanzamientos por año

Cantidad de satélites activos hoy que se lanzaron cada año desde 2015. Es una estimación: no incluye los satélites que ya dejaron de funcionar.

In [3]:
from catalogo import lanzamientos_por_anio

por_anio = lanzamientos_por_anio(registros)
for anio in range(2015, 2027):
    print(f"{anio}: {por_anio.get(anio, 0)}")

anio_max = 2015
for anio in por_anio:
    if anio >= 2015 and por_anio[anio] > por_anio[anio_max]:
        anio_max = anio
print(f"Año con más lanzamientos desde 2015: {anio_max} ({por_anio[anio_max]} satélites)")

2015: 96
2016: 83
2017: 136
2018: 194
2019: 145
2020: 360
2021: 994
2022: 1718
2023: 2266
2024: 2429
2025: 4289
2026: 3285
Año con más lanzamientos desde 2015: 2025 (4289 satélites)


**Lectura:** la cantidad de satélites lanzados por año creció muchísimo: de 96 en 2015 a 4.289 en 2025, que fue el año con más lanzamientos. El salto fuerte empieza después de 2020, y coincide con el despliegue de las constelaciones grandes. 2026 todavía no terminó, por eso su cantidad (3.285) es parcial. Como el archivo solo tiene los satélites activos hoy, los años más viejos están subestimados.

## 4. Tipos de órbita

Clasificación de los satélites según su altitud en el momento del cálculo: LEO (baja), MEO (media), GEO (geoestacionaria) y HEO (alta).

In [4]:
from analisis import contar_por_tipo

conteo_tipos = contar_por_tipo(satelites)
for tipo, cantidad in conteo_tipos.items():
    print(f"{tipo}: {cantidad}")

LEO: 15883
MEO: 207
GEO: 584
HEO: 20


**Lectura:** la gran mayoría de los satélites está en órbita baja (LEO), alrededor del 95 % del total, lo que es esperable porque Starlink y las otras constelaciones orbitan a pocos cientos de kilómetros. Le siguen la órbita geoestacionaria (GEO), con unos 585 satélites, la media (MEO) con unos 200, y muy pocos en órbita alta (HEO), unos 20. Estos valores cambian levemente en cada ejecución porque los satélites se mueven.

## 5. Extremos

Se busca el satélite más bajo y el más alto, y se listan los diez más bajos y los diez más altos.

In [5]:
from analisis import mas_bajo, mas_alto, top_n

print("Satélite más bajo:", mas_bajo(satelites))
print("Satélite más alto:", mas_alto(satelites))

print("\nTop 10 más bajos:")
for s in top_n(satelites, 10, mas_altos=False):
    print(" -", s)

print("\nTop 10 más altos:")
for s in top_n(satelites, 10, mas_altos=True):
    print(" -", s)

Satélite más bajo: STARLINK-38410 | LEO | 119.1 km
Satélite más alto: TESS | HEO | 366492.6 km

Top 10 más bajos:
 - STARLINK-38410 | LEO | 119.1 km
 - STARLINK-1641 | LEO | 119.9 km
 - COSMOS 2572 | LEO | 142.7 km
 - STARLINK-4454 | LEO | 159.4 km
 - STARLINK-38414 | LEO | 164.5 km
 - STARLINK-30924 | LEO | 168.7 km
 - STARLINK-30224 | LEO | 171.2 km
 - STARLINK-36255 | LEO | 171.8 km
 - IPERDRONE.0 | LEO | 173.4 km
 - STARLINK-30236 | LEO | 188.7 km

Top 10 más altos:
 - TESS | HEO | 366492.6 km
 - CXO | HEO | 126419.1 km
 - MMS 2 | HEO | 108034.6 km
 - MMS 1 | HEO | 108011.5 km
 - MMS 3 | HEO | 107983.9 km
 - MMS 4 | HEO | 107958.3 km
 - XMM-NEWTON | HEO | 90379.0 km
 - THEMIS A | HEO | 74916.9 km
 - SMILE | HEO | 72390.8 km
 - THEMIS D | HEO | 65526.2 km


**Lectura:** el satélite más alto es TESS, a más de 360.000 km de altura, y los diez más altos son todos de órbita alta (HEO): misiones científicas como TESS, CXO, MMS y XMM-Newton, que orbitan muy lejos de la Tierra. En el otro extremo, los diez más bajos están entre unos 120 y 190 km, y casi todos son Starlink. Esa altura es mucho menor que los cerca de 500 km a los que operan normalmente, así que probablemente sean satélites en proceso de caída hacia la atmósfera, muy cerca del límite de 100 km con el que se descartan.

## 6. Comparación de grupos

Tabla con el resumen de altitudes (cantidad, mínima, media y máxima, en km) de cada grupo.

In [6]:
from analisis import resumen_por_grupo

resumen = resumen_por_grupo(satelites)

print(f"{'Grupo':<12}{'Cantidad':>10}{'Mínima (km)':>14}{'Media (km)':>13}{'Máxima (km)':>14}")
for grupo_nombre, datos in resumen.items():
    print(f"{grupo_nombre:<12}{datos['cantidad']:>10}{datos['minima']:>14.1f}{datos['media']:>13.1f}{datos['maxima']:>14.1f}")

Grupo         Cantidad   Mínima (km)   Media (km)   Máxima (km)
Otros             4070         142.7       6414.2      366492.6
GPS                 40       19744.8      20216.4       21021.7
Globalstar          36         689.4       1295.4        1834.8
BeiDou              55       21502.1      27214.8       36055.0
Iridium             80         641.5        777.6         807.4
OneWeb             651         547.3       1210.0        1252.3
Starlink         11114         119.1        468.6         596.0
Qianfan            257         789.8       1043.1        1233.8
Kuiper             391         441.9        632.1         658.0


**Lectura:** cada grupo ocupa una franja de altitud distinta. Starlink es el más bajo (media de unos 469 km), seguido por Kuiper (unos 632 km) e Iridium (unos 778 km). Qianfan (unos 1.043 km), OneWeb (unos 1.210 km) y Globalstar (unos 1.295 km) orbitan más alto, pero siguen en órbita baja. GPS (unos 20.200 km) y BeiDou (unos 27.200 km) están en órbita media, y BeiDou llega hasta unos 36.000 km porque incluye satélites geoestacionarios. El grupo "Otros" tiene una media muy alta (unos 6.400 km) porque mezcla satélites de todo tipo, incluidos los más lejanos del catálogo.

## 7. Bariloche

Se buscan los satélites que están sobre la zona `BARILOCHE` (un cuadro de 10° x 10° centrado en la ciudad) en el momento del cálculo, y se los lista de menor a mayor altitud.

In [7]:
from analisis import en_zona, ordenar_por

BARILOCHE = (-76.30, -46.13, -66.30, -36.13)   # lon_min, lat_min, lon_max, lat_max

sobre_bariloche = en_zona(satelites, BARILOCHE)
print(f"Satélites sobre la zona de Bariloche: {len(sobre_bariloche)}\n")

for s in ordenar_por(sobre_bariloche, "altitud_km"):
    print(s)

Satélites sobre la zona de Bariloche: 43

STARLINK-11699 [DTC] | LEO | 351.5 km
STARLINK-11188 [DTC] | LEO | 373.0 km
STARLINK-1559 | LEO | 421.8 km
STARLINK-1392 | LEO | 423.6 km
STARLINK-1983 | LEO | 466.6 km
STARLINK-37562 | LEO | 472.6 km
STARLINK-35147 | LEO | 472.8 km
STARLINK-32853 | LEO | 473.2 km
STARLINK-37523 | LEO | 473.5 km
STARLINK-35128 | LEO | 473.6 km
STARLINK-33533 | LEO | 474.1 km
STARLINK-31317 | LEO | 474.9 km
STARLINK-31983 | LEO | 475.1 km
STARLINK-32039 | LEO | 476.2 km
STARLINK-34331 | LEO | 476.6 km
STARLINK-3176 | LEO | 477.6 km
STARLINK-32009 | LEO | 478.4 km
STARLINK-32050 | LEO | 478.5 km
STARLINK-33656 | LEO | 481.4 km
STARLINK-32243 | LEO | 484.0 km
STARLINK-5001 | LEO | 486.2 km
STARLINK-6112 | LEO | 493.6 km
STARLINK-35292 | LEO | 493.6 km
STARLINK-33697 | LEO | 493.6 km
STARLINK-36475 | LEO | 494.2 km
STARLINK-6166 | LEO | 494.5 km
STARLINK-31549 | LEO | 494.6 km
STARLINK-31873 | LEO | 495.3 km
STARLINK-6335 | LEO | 495.8 km
STARLINK-6369 | LEO | 496.

**Lectura:** en el momento de esta ejecución hay 43 satélites sobre la zona de Bariloche. Todos están en órbita baja, entre unos 350 y 1.500 km, y la gran mayoría (34 de 43) son Starlink, con altitudes cercanas a los 475-495 km. También aparecen algunos satélites de OneWeb, Kuiper y otros. El número cambia en cada ejecución, porque los satélites se mueven a varios km por segundo y salen y entran de la zona todo el tiempo.

## 8. Conclusiones

El catálogo analizado tiene 16.710 satélites, con órbitas medidas hace unos 12 días en promedio, por lo que las posiciones calculadas son confiables. El grupo que predomina es Starlink, con 11.125 satélites, alrededor de dos tercios del total (66,6 %). El año en que más satélites se lanzaron (de los que siguen activos) fue 2025, con 4.289, en un crecimiento muy fuerte desde 2020. El tipo de órbita que predomina es la órbita baja (LEO), con cerca del 95 % de los satélites, mientras que GEO, MEO y HEO suman muy pocos. Sobre la zona de Bariloche había 43 satélites en el momento de esta ejecución, todos en órbita baja y en su mayoría Starlink. Hay que recordar que los lanzamientos por año son una estimación, porque el archivo solo incluye satélites activos, y que algunos números (como los de tipo de órbita y los de Bariloche) cambian levemente en cada ejecución.